In [3]:
import os
import cv2
import pickle
import mediapipe as mp

In [29]:
def bounding_box(width_frame, height_frame, x_landmarks, y_landmarks):
    xmin, xmax = min(x_landmarks), max(x_landmarks)
    ymin, ymax = min(y_landmarks), max(y_landmarks)
    width, height = xmax - xmin, ymax - ymin
    if width > height:
        delta_x = 0.1 * width
        delta_y = delta_x + (width - height) / 2
    else:
        delta_y = 0.1 * height
        delta_x = delta_y + (height - width) / 2
    start_x = max(int((xmin - delta_x) * width_frame), 0)
    start_y = max(int((ymin - delta_y) * height_frame), 0)
    end_x   = min(int((xmax + delta_x) * width_frame), width_frame)
    end_y   = min(int((ymax + delta_y) * height_frame), height_frame)
    landmarks_norm = []
    for i in range(len(x_landmarks)):
        x = x_landmarks[i]
        y = y_landmarks[i]
        cx, cy = int(x * width_frame), int(y * height_frame)
        x_norm = (cx - start_x) / (end_x - start_x)
        y_norm = (cy - start_y) / (end_y - start_y)
        landmarks_norm.append([x_norm, y_norm])

    result = {'landmarks': landmarks_norm,
              'bounding_box_start': (start_x, start_y),
              'bounding_box_end': (end_x, end_y)}
    return result

In [15]:
"""
Mô phỏng qua camera

Nếu tay trái => tay phải
Nếu tay phải => tay phải

tay trái ngoài thật là tay trái khi nhận diện, ngược lại
"""

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

DATA = []
LABELS = []

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as hands:

    while True:
        x_ = []
        y_ = []
        ret, frame = cap.read()
        H, W, _ = frame.shape
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)

        if results.multi_hand_landmarks and results.multi_handedness:
            # for hand_landmarks in results.multi_hand_landmarks:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                label = handedness.classification[0].label  # "Left" hoặc "Right"

                x_landmarks = []
                y_landmarks = []
                hand_data = []
                if label == "Left":
                    for lm in hand_landmarks.landmark:
                        x_landmarks.append(1- lm.x)
                        y_landmarks.append(lm.y)
                        hand_data.append([1- lm.x, lm.y])
                else:
                    for lm in hand_landmarks.landmark:
                        x_landmarks.append(lm.x)
                        y_landmarks.append(lm.y)
                        hand_data.append([lm.x, lm.y])

                if len(x_landmarks) + len(y_landmarks) == 42:
                    output_bounding_box = bounding_box(W, H, x_landmarks, y_landmarks)
                    landmarks_norm = output_bounding_box['landmarks']
                    DATA.append(landmarks_norm)
                    (start_x, start_y) = output_bounding_box['bounding_box_start']
                    (end_x, end_y) = output_bounding_box['bounding_box_end']

                    cv2.rectangle(frame, (start_x, start_y), (end_x, end_y), (0, 255, 0), 2)
                    cv2.putText(frame, label, (start_x, start_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.imshow("Hands with Bounding Box", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()

In [32]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

DATA = []
LABELS = []
error_images= []

CLASSES = ['E', 'Y', 'M', 'X', 'N', 'R', 'P', 'C', 'O', 'del', 'T',
           'space', 'G', 'I', 'F', 'S', 'Q', 'V', 'K', 'U', 'A', 'B', 'L',
           'W', 'D', 'H']

with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as hands:

    for _class in CLASSES:
        for img in os.listdir(f'dataset/{_class}'):
            frame = cv2.imread(f'dataset/{_class}/{img}')
            if frame is None:
                print(f'Không đọc được ảnh: dataset/{_class}/{img}')
                error_images.append(f'dataset/{_class}/{img}')
                continue

            x_ = []
            y_ = []
            H, W, _ = frame.shape
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(frame_rgb)

            if results.multi_hand_landmarks and results.multi_handedness:
                for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                    label = handedness.classification[0].label  # "Left" hoặc "Right"

                    x_landmarks = []
                    y_landmarks = []
                    hand_data = []

                    if label == "Left":
                        for lm in hand_landmarks.landmark:
                            x_landmarks.append(1- lm.x)
                            y_landmarks.append(lm.y)
                            hand_data.append([1- lm.x, lm.y])
                    else:
                        for lm in hand_landmarks.landmark:
                            x_landmarks.append(lm.x)
                            y_landmarks.append(lm.y)
                            hand_data.append([lm.x, lm.y])

                    if len(x_landmarks) + len(y_landmarks) == 42:
                        output_bounding_box = bounding_box(W, H, x_landmarks, y_landmarks)
                        landmarks_norm = output_bounding_box['landmarks']
                        DATA.append(landmarks_norm)
                        LABELS.append(CLASSES.index(_class))
            else:
                print(f'dataset/{_class}/{img}')


dataset/E/E13_jpg.rf.8d68f0f191144ef3272f596f3b3ce637.jpg
dataset/E/E14_jpg.rf.099b7c532a31aed9e59b72301ea13303.jpg
dataset/E/E15_jpg.rf.6cea2b32a4805735fe5fed0e95ef23e8.jpg
dataset/E/E24_jpg.rf.e0373ac336a3057e59db86a7be244989.jpg
dataset/E/E_f417eaeb-2afa-11f0-90e5-0024d6f1f569.jpg
dataset/E/E_f418b4a9-2afa-11f0-871a-0024d6f1f569.jpg
dataset/E/E_f470cf86-2afa-11f0-9bf9-0024d6f1f569.jpg
dataset/E/E_f47171ba-2afa-11f0-875c-0024d6f1f569.jpg
dataset/E/E_f48445dc-2afa-11f0-b05e-0024d6f1f569.jpg
dataset/E/E_f48509a4-2afa-11f0-abff-0024d6f1f569.jpg
dataset/E/E_f4ac1427-2afa-11f0-9795-0024d6f1f569.jpg
dataset/E/E_f4b1c078-2afa-11f0-b80e-0024d6f1f569.jpg
dataset/E/E_f4b25ca0-2afa-11f0-9deb-0024d6f1f569.jpg
dataset/E/E_f4b3e7be-2afa-11f0-92e8-0024d6f1f569.jpg
dataset/E/E_f4b48439-2afa-11f0-81df-0024d6f1f569.jpg
dataset/E/E_f4b525a6-2afa-11f0-be34-0024d6f1f569.jpg
dataset/E/E_f4b60386-2afa-11f0-8bc5-0024d6f1f569.jpg
dataset/E/E_f4b69fbd-2afa-11f0-97eb-0024d6f1f569.jpg
dataset/E/E_f4bf201b-2afa-

In [15]:
with open('DATASET.pickle', 'wb') as f:
    pickle.dump({'data': DATA, 'labels': LABELS}, f)

In [16]:
with open('DATASET.pickle', 'rb') as f:
    dataset = pickle.load(f)

# Lấy lại dữ liệu
DATA = dataset['data']
LABELS = dataset['labels']

In [18]:
print(DATA.shape)
print(LABELS.shape)

torch.Size([61940, 21, 2])
torch.Size([61940])
